<a href="https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/14_capstone_challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- nav-header -->
[⬅ Previous](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/13_fairness_and_subgroups.ipynb) · [🗺️ Course index](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/00_START_HERE.ipynb) · **Notebook 14 of the course** · [Next: 15 — Capstone solutions ➡](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/15_capstone_solutions.ipynb)


# 🏆 Notebook 14 — The capstone challenge

**A friendly competition. You use everything you learned today, and we compare results.**

---

## What you have to do

Predict, for each patient, **how likely it is that they die within 90 days**.

We have divided the patients into two groups:

| Group | Size | What you may do with it |
|-------|-----:|-------------------------|
| **Training patients** | ~1,270 | Everything. Build features, train models, try ideas. |
| **Held-out patients** | ~420 | Only make predictions for them. Never learn from them. |

*"Held-out"* simply means: we hide these patients from the model while it learns. We then check the
predictions against what really happened. This is how we find out whether a model works on **new**
patients, which is the only thing that matters in practice.

When you are ready, you call `score(...)`. It compares your predictions with the true outcomes and
puts your result on a leaderboard.

**Everyone in the room gets exactly the same two groups**, so the leaderboard is a fair comparison.

---

## The rules

| ✅ You may | ❌ You may not |
|-----------|---------------|
| build any features you like from the training patients | use the column `died_in_hosp` — it already tells you the answer (see Notebook 08) |
| use any model and any library | let the held-out patients influence *anything* your model learns |
| test your ideas with cross-validation (explained below) | submit again and again until the number looks nice |
| combine several models | put the same patient in both groups |

---

## What score to expect

- The example model further down reaches **0.754**. This number is the **ROC-AUC**; it is explained
  in a moment. Higher is better, 0.5 means "no better than guessing".
- The worked solutions in **Notebook 15** reach about **0.78**. That is the realistic target. We
  measured it, we did not guess it.
- So the whole range you are competing in is roughly **0.75 to 0.78**. Improvements in this field are
  small. A gain of +0.01 is a normal, respectable result.
- Above **0.93**, something is wrong. Almost certainly your model has seen the answer somewhere.
  Finding that mistake is more valuable than winning, so please tell the room if it happens.

**⏱️ Time:** 45–60 minutes · **You should have done first:** Notebooks 04, 05 and 06.
Notebooks 08 and 09 also help.

## 📖 A short glossary

These words appear below. Here is what each one means, in one line.

| Word | Meaning |
|------|---------|
| **feature** | one number that describes a patient, e.g. "highest lactate during the stay" |
| **model** | the thing that turns features into a prediction |
| **to train / to fit** | to let the model learn from data |
| **ROC-AUC** | "if I pick one patient who died and one who survived, how often does the model give the higher risk to the one who died?" 1.0 = always, 0.5 = coin flip |
| **sensitivity** | of all the patients who died, what share did we correctly flag? |
| **calibration** | if the model says "20% risk", do about 20 out of 100 such patients really die? |
| **Brier score** | one number combining accuracy and calibration. **Lower is better** here |
| **cross-validation (CV)** | split the *training* patients into 5 parts, train on 4, test on 1, repeat 5 times, average. A safe way to test an idea without touching the held-out patients |
| **leakage** | information about the answer sneaks into the features by mistake, so the score looks great but the model is useless |
| **confidence interval** | the range in which the true value probably lies. Wide interval = we are not sure |

## ⚙️ Run this cell first

In [1]:
# === ⚙️  Workshop setup — run this cell first ===============================
# Works in Google Colab and in local Jupyter. Installs anything missing, sets a
# clean plotting style, and gives you helpers to load the data.
# (This cell is identical in every notebook of the course.)
import importlib.util, subprocess, sys, os, random, warnings
warnings.filterwarnings("ignore")

# --- reproducibility: everyone in the room gets the same numbers ---------------
RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)

# 📥 Where the workshop data comes from — already set up for you, nothing to do.
# The data downloads automatically the first time you need it. If you were given a
# different link, just paste it in place of the one below. These forms all work:
#   • a Google-Drive folder link      • a Drive / Dropbox / OneDrive file link
#   • a folder URL ending in "/"      • a link straight to a .zip
# Set it to "" if you would rather upload the CSVs by hand.
# (The data is not in the GitHub repo: it is real de-identified patient data covered
#  by a data use agreement and may not be redistributed openly.)
WORKSHOP_DATA_URL = os.environ.get(
    "WORKSHOP_DATA_URL",
    "https://drive.google.com/drive/folders/1y7CparhrqdlCAZq6xQlj8fZda394fniD")

def _ensure(pkgs):
    missing = [pip for mod, pip in pkgs.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", *missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing])
_ensure({"numpy":"numpy","pandas":"pandas","sklearn":"scikit-learn",
         "matplotlib":"matplotlib","seaborn":"seaborn","shap":"shap","xgboost":"xgboost"})

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
np.random.seed(RANDOM_STATE)          # seeds the legacy global np.random.* calls
RNG = np.random.default_rng(RANDOM_STATE)   # the modern generator — use this one
pd.set_option("display.max_columns", 120); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5); plt.rcParams["figure.dpi"] = 110

# Every model, split and resample in this course passes random_state=RANDOM_STATE, so
# your numbers should match your neighbour's exactly. (Different library *versions* can
# still shift the last decimal — that is normal and not a mistake on your part.)

# --- data loading: works locally AND remembers your upload across notebooks -----
_CACHE = {"dir": "unset"}   # memo so we only touch Google Drive once per session

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _drive_cache():
    """In Google Colab, mount Drive ONCE and return a persistent folder. A file you
    upload in one notebook is saved here, so every other notebook opens it automatically
    — no re-uploading. Returns None outside Colab, or if you decline to connect Drive."""
    if _CACHE["dir"] != "unset":
        return _CACHE["dir"]
    result = None
    if _in_colab():
        try:
            from google.colab import drive
            if not os.path.ismount("/content/drive"):
                drive.mount("/content/drive")
            result = "/content/drive/MyDrive/sepsis_workshop_data"
            os.makedirs(result, exist_ok=True)
        except Exception:
            result = None
    _CACHE["dir"] = result
    return result

def _find(name):
    """Look for the file on disk. Deliberately does NOT touch Google Drive, so the normal
    path never triggers an authorisation popup."""
    for p in [name, f"data/{name}", f"../data/{name}", f"workshop/data/{name}"]:
        if os.path.exists(p):
            return p
    return None

def _find_in_drive(name):
    """Only used as a fallback, because it mounts Drive (and that means a popup)."""
    cache = _drive_cache()
    if cache:
        p = os.path.join(cache, name)
        if os.path.exists(p):
            return p
    return None

def _direct_url(u):
    """Turn an ordinary Google-Drive / Dropbox / OneDrive *share* link into one that a
    plain HTTP client can actually download, so you can paste the link you were given."""
    import re
    m = (re.search(r"drive\.google\.com/file/d/([\w-]+)", u)
         or re.search(r"drive\.google\.com/(?:open|uc)\?(?:export=\w+&)?id=([\w-]+)", u))
    if m:
        return f"https://drive.google.com/uc?export=download&id={m.group(1)}"
    if "dropbox.com" in u:
        return u.split("?")[0] + "?dl=1"
    if "sharepoint.com" in u or "1drv.ms" in u:
        return u + ("&" if "?" in u else "?") + "download=1"
    return u

def _fetch(url, dest):
    import urllib.request, shutil as _sh
    req = urllib.request.Request(_direct_url(url), headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
        _sh.copyfileobj(r, f)

_FOLDER = {"done": False}

def _gdrive_folder(name):
    """WORKSHOP_DATA_URL points at a Google-Drive *folder*: fetch it once with gdown
    (a folder cannot be downloaded with a plain HTTP request), then serve files from it."""
    dest = "_workshop_data"
    if not _FOLDER["done"]:
        _ensure({"gdown": "gdown"})
        import gdown
        print("⬇  fetching the workshop data from Google Drive (just once) …")
        gdown.download_folder(url=WORKSHOP_DATA_URL, output=dest, quiet=True, use_cookies=False)
        _FOLDER["done"] = True
    for root, _dirs, files in os.walk(dest):
        if name in files:
            return os.path.join(root, name)
    return None

def _try_download(name):
    """Fetch the data from WORKSHOP_DATA_URL, if one was configured."""
    u = (WORKSHOP_DATA_URL or "").strip()
    if not u:
        return None
    try:
        if "/drive/folders/" in u:
            return _gdrive_folder(name)
        if u.lower().split("?")[0].endswith(".zip"):
            import zipfile
            bundle = "_workshop_data.zip"
            if not os.path.exists(bundle):
                print("⬇  downloading the workshop data bundle …")
                _fetch(u, bundle)
            with zipfile.ZipFile(bundle) as z:      # flatten any folder inside the zip
                for member in z.namelist():
                    if os.path.basename(member) == name:
                        with z.open(member) as src, open(name, "wb") as dst:
                            dst.write(src.read())
                        return name
            print(f"  ({name} was not inside the bundle)")
            return None
        print(f"⬇  downloading {name} …")
        _fetch(u.rstrip("/") + "/" + name, name)
        return name
    except Exception as e:
        print(f"  (download failed: {e})")
        for leftover in (name, "_workshop_data.zip"):
            if os.path.exists(leftover) and os.path.getsize(leftover) == 0:
                os.remove(leftover)
        return None

def _cache_to_drive(name, data):
    cache = _drive_cache()
    if cache:
        dest = os.path.join(cache, name)
        data.to_csv(dest, index=False)
        print(f"  💾 saved to Google Drive ({dest}) — no need to fetch it again.")

def load_csv(name):
    """Load a data CSV: looks on disk, then downloads it from WORKSHOP_DATA_URL, then checks
    your Google-Drive cache, and only as a last resort asks you to upload it — in which case
    it saves a copy to Drive so you never have to upload it twice."""
    p = _find(name)                       # 1. already on disk?
    if p:
        print(f"✓ loaded {p}")
        return pd.read_csv(p)
    p = _try_download(name)               # 2. the built-in download link
    if p:
        print(f"✓ loaded {name}")
        return pd.read_csv(p)
    p = _find_in_drive(name)              # 3. a copy you saved on a previous run
    if p:
        print(f"✓ loaded {p}")
        return pd.read_csv(p)
    try:                                  # 4. last resort: upload it by hand
        from google.colab import files
        print(f"⤴  Upload {name} just once — I'll save it so the other notebooks open it automatically:")
        up = files.upload()
        fname = list(up.keys())[0]
        data = pd.read_csv(fname)
        _cache_to_drive(name, data)
        return data
    except Exception:
        raise FileNotFoundError(
            f"Could not find {name}. Either paste your download link into WORKSHOP_DATA_URL at "
            f"the top of this cell, or put the CSV next to this notebook / in a data/ folder."
        )


## 📂 Load the data and create the two groups

We divide the patients **by patient ID**, never row by row. One patient produces many rows (one row
per 4 hours). If some of a patient's rows were in the training group and others in the held-out
group, the model would already know that patient — and the score would be far too good. Notebook 08
shows exactly that mistake.

We fix the random seed, so everyone gets the same two groups.

In [2]:
def build_patient_table(ts):
    """Aggregate the 4-hourly time-series into ONE row per ICU stay.
       (This is exactly what Notebook 04 teaches you to build.)"""
    df = ts.copy()
    # clean temperature: prefer Celsius; repair obvious Fahrenheit-entry errors; drop impossible
    tf = df["Temp_F"].where((df["Temp_F"] >= 90) & (df["Temp_F"] <= 110))
    tc = df["Temp_C"].where((df["Temp_C"] >= 25) & (df["Temp_C"] <= 45))
    df["Temp_C_clean"] = tc.fillna((tf - 32) * 5/9)
    # impossible vitals -> NaN, then forward/back-fill WITHIN each stay
    for c, lo, hi in [("HR",20,300),("SysBP",40,300),("MeanBP",20,220),("RR",3,80),("SpO2",30,100)]:
        df[c] = df[c].where((df[c] >= lo) & (df[c] <= hi))
    g = df.groupby("icustayid", group_keys=False)
    vit = ["HR","SysBP","MeanBP","RR","SpO2","Temp_C_clean"]
    df[vit] = g[vit].apply(lambda x: x.ffill().bfill())
    # winsorise skewed labs so one stray value can't define a min/max
    for c in ["Arterial_lactate","Creatinine","BUN","WBC_count","Platelets_count","INR","Glucose"]:
        lo, hi = df[c].quantile(0.01), df[c].quantile(0.99)
        df[c] = df[c].clip(lo, hi)
    tv = ["HR","SysBP","MeanBP","RR","SpO2","Temp_C_clean","GCS","Creatinine","BUN",
          "Arterial_lactate","WBC_count","Platelets_count","Potassium","Sodium",
          "Albumin","INR","Arterial_pH","Shock_Index","SOFA","SIRS","PaO2_FiO2","Hb"]
    grp = df.groupby("icustayid")
    f = {}
    f["age"]=grp["age"].first(); f["gender"]=grp["gender"].first()
    f["elixhauser"]=grp["elixhauser"].first(); f["re_admission"]=grp["re_admission"].first().astype(int)
    f["weight_kg"]=grp["Weight_kg"].median(); f["n_blocs"]=grp.size(); f["los_hours"]=grp["bloc"].max()*4
    f["mechvent_ever"]=grp["mechvent"].max(); f["vaso_max"]=grp["max_dose_vaso"].max()
    f["fluid_balance_last"]=grp["cumulated_balance"].last(); f["urine_total"]=grp["output_step"].sum()
    for c in tv:
        f[f"{c}_mean"]=grp[c].mean(); f[f"{c}_min"]=grp[c].min()
        f[f"{c}_max"]=grp[c].max();  f[f"{c}_last"]=grp[c].last()
    for c in ["Creatinine","Arterial_lactate","SOFA","Shock_Index","GCS"]:
        f[f"{c}_delta"]=grp[c].last()-grp[c].first()
    X = pd.DataFrame(f); X["morta_90"]=grp["morta_90"].max(); X["died_in_hosp"]=grp["died_in_hosp"].max()
    return X.reset_index()

def load_patients():
    p = _find("sepsis_patients.csv")
    if p:
        print(f"✓ loaded {p}"); return pd.read_csv(p)
    print("sepsis_patients.csv not found — building it from the time-series file…")
    return build_patient_table(load_csv("sepsis_timeseries.csv"))

In [3]:
ts = load_csv("sepsis_timeseries.csv")

# ---- create the two groups: by patient, fixed seed, the same for everybody ----
from sklearn.model_selection import train_test_split

stay_outcome = ts.groupby("icustayid")["morta_90"].max()
TRAIN_IDS, TEST_IDS = train_test_split(
    stay_outcome.index.to_numpy(), test_size=0.25,
    stratify=stay_outcome.to_numpy(), random_state=2026)
TRAIN_IDS, TEST_IDS = set(TRAIN_IDS.tolist()), set(TEST_IDS.tolist())

ts_train = ts[ts["icustayid"].isin(TRAIN_IDS)].copy()
ts_test  = ts[ts["icustayid"].isin(TEST_IDS)].copy()

# The true outcomes of the held-out patients. You may look at them, but do not let
# your model learn from them - that would only fool yourself.
Y_TEST = ts_test.groupby("icustayid")["morta_90"].max()

print(f"training patients : {len(TRAIN_IDS):,}  ({ts_train.shape[0]:,} rows)")
print(f"held-out patients : {len(TEST_IDS):,}  ({ts_test.shape[0]:,} rows)")
print(f"died within 90 days - training {stay_outcome[list(TRAIN_IDS)].mean():.1%} · "
      f"held-out {Y_TEST.mean():.1%}   (these should be close)")
assert not (TRAIN_IDS & TEST_IDS), "a patient is in both groups!"

✓ loaded data/sepsis_timeseries.csv


training patients : 1,272  (27,995 rows)
held-out patients : 424  (9,709 rows)
died within 90 days - training 18.2% · held-out 18.2%   (these should be close)


## 🧮 How your result is measured

`score(predictions, name)` needs **one number per held-out patient**: the probability that this
patient dies within 90 days. A value between 0 and 1.

It then prints four numbers:

| Number | Question it answers | Better when |
|--------|--------------------|-------------|
| **ROC-AUC** | Does the model put the sicker patient higher? | higher |
| **Average precision** | Same idea, but fairer when deaths are rare (here only 18%) | higher |
| **Brier score** | Are the probabilities themselves believable? | **lower** |
| **Sensitivity at 10% alerts** | If the ward can look after only the 10% highest-risk patients, how many of the deaths do we catch? | higher |

It also prints a **95% confidence interval** for the ROC-AUC. This matters a lot here.

We only have 424 held-out patients, so the interval is wide — about **±0.06**. If your score is 0.77
and the example model is 0.754, but the two intervals overlap, then **you have not really won**. You
have seen random variation. This is the same lesson as Notebook 13, now applied to your own result.

Note what this implies: the interval (±0.06) is *wider than the whole range worth competing in*
(0.75 to 0.78). Take the leaderboard in the friendly spirit it is meant.

Finally, `score` refuses obviously broken submissions (missing patients, empty values, numbers
outside 0–1).

In [4]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

LEADERBOARD = []
_rng = np.random.default_rng(RANDOM_STATE)

def _auc_ci(y, p, n_boot=400):
    """95% confidence interval for the ROC-AUC.

    Method: draw a random sample of the patients (with repetition) 400 times and
    re-compute the score each time. The spread of those 400 scores tells us how
    much the result depends on which patients we happened to have."""
    y, p = np.asarray(y), np.asarray(p)
    aucs = []
    for _ in range(n_boot):
        i = _rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) > 1:
            aucs.append(roc_auc_score(y[i], p[i]))
    return float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))

def score(preds, name="my model", verbose=True):
    """Measure a submission.

    preds = one probability of death per held-out patient (a pandas Series whose
            index is icustayid). Values must be between 0 and 1."""
    preds = pd.Series(preds).astype(float)
    missing = set(Y_TEST.index) - set(preds.index)
    if missing:
        raise ValueError(f"{len(missing)} held-out patients have no prediction (e.g. {list(missing)[:3]})")
    if preds.isna().any():
        raise ValueError("your predictions contain empty values (NaN)")
    if preds.min() < 0 or preds.max() > 1:
        raise ValueError("predictions must be probabilities between 0 and 1")
    p = preds.reindex(Y_TEST.index).clip(1e-6, 1 - 1e-6)

    k = max(1, int(round(0.10 * len(p))))              # the 10% highest-risk patients
    flagged = p.nlargest(k).index
    sens_at_10 = Y_TEST.loc[flagged].sum() / max(1, Y_TEST.sum())
    lo, hi = _auc_ci(Y_TEST, p)

    res = {"name": name,
           "ROC_AUC": roc_auc_score(Y_TEST, p),
           "AUC_lo": lo, "AUC_hi": hi,
           "avg_precision": average_precision_score(Y_TEST, p),
           "brier": brier_score_loss(Y_TEST, p),
           "sens_at_10pct_alerts": float(sens_at_10)}
    LEADERBOARD.append(res)
    if verbose:
        print(f"📊 {name}")
        print(f"   ROC-AUC ................. {res['ROC_AUC']:.4f}   [95% CI {lo:.3f}–{hi:.3f}]")
        print(f"   Average precision ....... {res['avg_precision']:.4f}   (18% of patients die)")
        print(f"   Brier score (lower=better) {res['brier']:.4f}")
        print(f"   Deaths caught in top 10%  {res['sens_at_10pct_alerts']:.1%}")
        if res["ROC_AUC"] > 0.93:
            print("   🚨 This is too good to be true. Look for leakage first (Notebook 08).")
    return res

def leaderboard():
    return (pd.DataFrame(LEADERBOARD)
              .sort_values("ROC_AUC", ascending=False)
              .reset_index(drop=True).round(4))

def board_score(name):
    """Find an earlier submission again by its name."""
    return next(r for r in reversed(LEADERBOARD) if r["name"] == name)

## 🥉 Comparison 1 — the "no model at all" model

Before building anything, find out what doing **nothing** scores. Here we give every patient the same
number: the average death rate.

This must land at ROC-AUC = 0.5, because such a prediction cannot tell any two patients apart.
It is the floor. Any real model has to be clearly above it.

In [5]:
score(pd.Series(Y_TEST.mean(), index=Y_TEST.index), "no model (same risk for everyone)");

📊 no model (same risk for everyone)
   ROC-AUC ................. 0.5000   [95% CI 0.500–0.500]
   Average precision ....... 0.1816   (18% of patients die)
   Brier score (lower=better) 0.1486
   Deaths caught in top 10%  13.0%


## 🥈 Comparison 2 — one single clinical value

Now something a doctor could do without any computer: rank patients by their **worst SOFA score**
during the stay. (SOFA measures how many organ systems are failing. Higher is worse.)

This is the number your model really has to beat. A clinical reviewer will ask exactly this:
*"Is your model better than the score we already use at the bedside?"*

In [6]:
sofa_max_test = ts_test.groupby("icustayid")["SOFA"].max()
score(sofa_max_test / sofa_max_test.max(), "worst SOFA score alone");

📊 worst SOFA score alone
   ROC-AUC ................. 0.6768   [95% CI 0.609–0.750]
   Average precision ....... 0.3519   (18% of patients die)
   Brier score (lower=better) 0.2022
   Deaths caught in top 10%  24.7%


## 🥇 The example model — this is what you try to beat

Nothing new here. It is the recipe from Notebook 05:

1. summarise each patient's whole stay into one row of features,
2. fill in missing values,
3. train a gradient-boosting model (XGBoost).

Everything is learned from the **training patients only**.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

def features_from(ts_slice):
    """Turn the 4-hourly rows into one row per patient.
       Each patient is summarised using only their own data."""
    tab = build_patient_table(ts_slice).set_index("icustayid")
    return tab.drop(columns=["morta_90", "died_in_hosp"])

Xtr = features_from(ts_train)
Xte = features_from(ts_test).reindex(columns=Xtr.columns)   # same columns, same order
ytr = ts_train.groupby("icustayid")["morta_90"].max().reindex(Xtr.index)

baseline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("xgb", XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
                          scale_pos_weight=float((ytr == 0).sum() / max(1, (ytr == 1).sum())),
                          random_state=RANDOM_STATE, n_jobs=-1)),
])
baseline.fit(Xtr, ytr)

pred_baseline = pd.Series(baseline.predict_proba(Xte)[:, 1], index=Xte.index)
score(pred_baseline, "example model (XGBoost)");

📊 example model (XGBoost)
   ROC-AUC ................. 0.7540   [95% CI 0.696–0.800]
   Average precision ....... 0.3966   (18% of patients die)
   Brier score (lower=better) 0.1462
   Deaths caught in top 10%  24.7%


In [8]:
leaderboard()

,name,ROC_AUC,AUC_lo,AUC_hi,avg_precision,brier,sens_at_10pct_alerts
0,example model (XGBoost),0.7540,0.6958,0.7998,0.3966,0.1462,0.2468
1,worst SOFA score alone,0.6768,0.6093,0.7497,0.3519,0.2022,0.2468
2,no model (same risk for everyone),0.5000,0.5000,0.5000,0.1816,0.1486,0.1299


## 🧪 Important: test your ideas *without* submitting

Do **not** try an idea, submit it, look at the score, change something, submit again, and repeat.

Why not? Because every time you look at the held-out score and then change your model, you are using
those patients to make a decision. After twenty rounds of this, your model is quietly tuned to those
420 patients, and the score no longer says anything about new patients. This is a slow, invisible
version of the leakage problem from Notebook 08.

**Do this instead:** test every idea with **cross-validation on the training patients**. That gives
you an honest comparison and costs you nothing. Submit only two or three times in the whole session.

The cell below shows both numbers for the example model, so you can see how they relate.

In [9]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_auc = cross_val_score(baseline, Xtr, ytr, cv=cv, scoring="roc_auc", n_jobs=-1)
print(f"Cross-validation on training patients : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
bl = board_score("example model (XGBoost)")
print(f"Score on the held-out patients        : {bl['ROC_AUC']:.4f}  "
      f"[95% CI {bl['AUC_lo']:.3f}-{bl['AUC_hi']:.3f}]")
print("\nThe two numbers should be reasonably close.")
print("If they are very different, the cross-validation is not testing what you think.")

Cross-validation on training patients : 0.7788 ± 0.0485
Score on the held-out patients        : 0.7540  [95% CI 0.696-0.800]

The two numbers should be reasonably close.
If they are very different, the cross-validation is not testing what you think.


## 💡 Ideas to try

These are sorted by **how much they usually help**. Most people try them in the opposite order, so
start at the top.

### 1. Better features — this helps most (Notebook 04)

- **Look at change over time, not just one value.**
  A lactate of 4 that is *falling* is very different from a lactate of 4 that is *rising*.
  A single "highest value" cannot see the difference. Try `last value − first value`, or the change
  over the last 24 hours (that is 6 blocks of 4 hours).
- **Use only the first 24 hours** (`bloc <= 6`) and see how much score you lose. That version could
  actually warn a ward in time, so the loss tells you the real price of being useful.
- **How much treatment was given:** total vasopressor dose, hours on a ventilator, fluid balance.
- **Missing values carry information.** Whether a lab test was *ordered at all* says something about
  how worried the team was. Try `df[col].isna().mean()` for each patient.
- **How unstable was the patient?** The standard deviation (`std()`) of heart rate, blood pressure or
  lactate. An average hides instability.

### 2. Cleaner data (Notebook 02, measured in Notebook 09)

Repair the Fahrenheit/Celsius mix-up, limit extreme laboratory values, and carry the last known
value forward **within one patient**.

### 3. A different model

- XGBoost, LightGBM and HistGradientBoosting all work well on this kind of table.
- Logistic regression with good features is often almost as accurate — and much easier to explain to
  a clinician. That is a real advantage, not a consolation prize.
- **Combine models:** take the average of the predictions of 2–3 different models. This is simple and
  usually helps a little.

### 4. Calibration

This improves the Brier score, but not the ROC-AUC. Wrap your model in
`CalibratedClassifierCV(..., method="isotonic", cv=5)`.

### 5. Model settings (hyperparameters) — do this last (Notebook 09)

Use `RandomizedSearchCV` with a limited number of tries. Expect roughly +0.005, not +0.05.

In [10]:
# ============================================================================
#  ✏️  YOUR MODEL - change anything you like below.
#  Build features from ts_train, train on the training patients only,
#  then predict for the held-out patients.
# ============================================================================

def my_features(ts_slice):
    """Start from the example features and add your own ideas."""
    tab = build_patient_table(ts_slice).set_index("icustayid")
    tab = tab.drop(columns=["morta_90", "died_in_hosp"])
    g = ts_slice.sort_values(["icustayid", "bloc"]).groupby("icustayid")

    # --- idea 1: how much did these values move up and down? ------------------
    for c in ["HR", "MeanBP", "Arterial_lactate", "SOFA"]:
        tab[f"{c}_std"] = g[c].std()

    # --- idea 2: the change over the LAST 24 hours (= 6 blocks of 4 hours) -----
    last24 = ts_slice[ts_slice["bloc"] > ts_slice.groupby("icustayid")["bloc"].transform("max") - 6]
    g24 = last24.groupby("icustayid")
    for c in ["Arterial_lactate", "SOFA", "Creatinine"]:
        tab[f"{c}_slope24"] = g24[c].last() - g24[c].first()

    # --- idea 3: how often was this test missing? -----------------------------
    for c in ["Arterial_lactate", "Albumin", "INR"]:
        tab[f"{c}_missing_frac"] = g[c].apply(lambda s: s.isna().mean())

    # 👉 add your own ideas here ...

    return tab

Xtr2 = my_features(ts_train)
Xte2 = my_features(ts_test).reindex(columns=Xtr2.columns)

my_model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("xgb", XGBClassifier(n_estimators=600, max_depth=4, learning_rate=0.04,
                          subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
                          eval_metric="logloss",
                          scale_pos_weight=float((ytr == 0).sum() / max(1, (ytr == 1).sum())),
                          random_state=RANDOM_STATE, n_jobs=-1)),
])

# --- test with cross-validation FIRST, before you submit ---------------------
cv2 = cross_val_score(my_model, Xtr2, ytr, cv=cv, scoring="roc_auc", n_jobs=-1)
print(f"My model, cross-validation : {cv2.mean():.4f} ± {cv2.std():.4f}")
print(f"Example model, same test   : {cv_auc.mean():.4f}")

My model, cross-validation : 0.7807 ± 0.0507
Example model, same test   : 0.7788


If your cross-validation number went up, it is worth submitting. Run the next cell.

If it did not go up, change something and test again — you have not used up a submission.

In [11]:
# --- only submit once cross-validation says you improved ---------------------
my_model.fit(Xtr2, ytr)
score(pd.Series(my_model.predict_proba(Xte2)[:, 1], index=Xte2.index), "my model v1");
leaderboard()

📊 my model v1
   ROC-AUC ................. 0.7548   [95% CI 0.695–0.810]
   Average precision ....... 0.3962   (18% of patients die)
   Brier score (lower=better) 0.1476
   Deaths caught in top 10%  27.3%


,name,ROC_AUC,AUC_lo,AUC_hi,avg_precision,brier,sens_at_10pct_alerts
0,my model v1,0.7548,0.6951,0.8103,0.3962,0.1476,0.2727
1,example model (XGBoost),0.7540,0.6958,0.7998,0.3966,0.1462,0.2468
2,worst SOFA score alone,0.6768,0.6093,0.7497,0.3519,0.2022,0.2468
3,no model (same risk for everyone),0.5000,0.5000,0.5000,0.1816,0.1486,0.1299


## 🤝 The room's leaderboard

Read your ROC-AUC out loud, or type your neighbours' numbers into the cell below to see everyone
together. Remember to compare the **confidence intervals**, not only the scores.

In [12]:
# Example of how to add somebody else's result:
# LEADERBOARD.append({"name": "Anna & Ben", "ROC_AUC": 0.79, "AUC_lo": 0.74, "AUC_hi": 0.84,
#                     "avg_precision": 0.44, "brier": 0.132, "sens_at_10pct_alerts": 0.30})
leaderboard()

,name,ROC_AUC,AUC_lo,AUC_hi,avg_precision,brier,sens_at_10pct_alerts
0,my model v1,0.7548,0.6951,0.8103,0.3962,0.1476,0.2727
1,example model (XGBoost),0.7540,0.6958,0.7998,0.3966,0.1462,0.2468
2,worst SOFA score alone,0.6768,0.6093,0.7497,0.3519,0.2022,0.2468
3,no model (same risk for everyone),0.5000,0.5000,0.5000,0.1816,0.1486,0.1299


## 🧠 Final discussion — the most important part

It does not matter very much who won. Please keep 10 minutes for these five questions. They are the
questions a journal reviewer, an ethics committee and a hospital director will ask you, in that order.

**1. Would your model still work in a different hospital?**
Other hospitals have other patients, other laboratory methods and other admission rules. The score
will drop. That is expected, not a failure — see the external validation part of Notebook 08.

**2. What would actually change because of it?**
A risk score that nobody acts on changes nothing. Who receives the alarm? What do they do? And how
many alarms per shift can a ward realistically handle?

**3. What did your best feature really measure?**
Use SHAP (Notebook 07) on your model. Suppose the strongest feature is "an arterial line was
placed". Then the model may have learned *that the team was worried*, not the patient's physiology.
Such a feature stops working as soon as clinical practice changes.

**4. Does it work equally well for everyone?**
Run your model through the subgroup analysis of Notebook 13 before you trust it. Good average
performance can hide poor performance in one group of patients.

**5. Are the probabilities believable?**
A model with a better ROC-AUC but a worse Brier score is often the **worse** choice for real use.
If it says 20%, roughly 20 out of 100 such patients should die.

> 🩺 **The honest summary:** careful feature engineering and model choice move this task from 0.754
> to about 0.777 (see Notebook 15 — every number there was measured). That gain is real, but it is
> **smaller than the ±0.06 uncertainty** of a 424-patient test group, so this dataset cannot even
> prove it. Turning any such model into something that helps a patient needs a prospective study, a
> clinical workflow, and a team. Today you practised the first part.

## ✏️ If you want to go further

- **Early warning.** Use only the first 24 hours of each stay (`ts[ts.bloc <= 6]`) and score again.
  How much do you lose? That difference is the price of predicting *early enough to act*.
- **A prediction at every time point.** Instead of one prediction per patient, predict at every
  4-hour block, using only the data available up to that moment. This is how a real bedside system
  works.
- **Beat XGBoost with logistic regression.** Same features, simpler model. If you come within 0.01,
  make the argument for using the model that can be explained.
- **A different competition.** Sort the leaderboard by **Brier score** instead of ROC-AUC. Does the
  winner change? Discuss which of the two competitions is the more useful one.

---

When you are finished, open **Notebook 15 — Capstone solutions** for worked answers to all of these,
with the measured scores.

<!-- nav-footer -->
---

### ➡️ Next up — Notebook 15: Capstone solutions

<a href="https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/15_capstone_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook 15 in Colab" height="32"/></a>

👉 **[Continue to Notebook 15 — Capstone solutions](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/15_capstone_solutions.ipynb)**

[⬅ Back to Notebook 13](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/13_fairness_and_subgroups.ipynb) · [🗺️ Course index](https://colab.research.google.com/github/lorenzkap/ML2026/blob/main/00_START_HERE.ipynb) · [📁 The course on GitHub](https://github.com/lorenzkap/ML2026)